In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

sns.set_theme(style="whitegrid")

In [2]:
# 1. Load Data
df = pd.read_csv('train.csv').copy()
target_col = 'Loan_Status'

# Drop ID column as it is unique and doesn't help prediction
if 'Loan_ID' in df.columns:
    df = df.drop('Loan_ID', axis=1)

print(f'Shape: {df.shape}')
print('\nFirst 5 rows:')
print(df.head())

print('\nData Types:')
print(df.dtypes)

print('\nSummary Statistics:')
print(df.describe(include='all').T)

print(f'\nDuplicate rows: {df.duplicated().sum()}')
print('\nMissing Values:')
print(df.isna().sum())

Shape: (614, 12)

First 5 rows:
  Gender Married Dependents     Education Self_Employed  ApplicantIncome  \
0   Male      No          0      Graduate            No             5849   
1   Male     Yes          1      Graduate            No             4583   
2   Male     Yes          0      Graduate           Yes             3000   
3   Male     Yes          0  Not Graduate            No             2583   
4   Male      No          0      Graduate            No             6000   

   CoapplicantIncome  LoanAmount  Loan_Amount_Term  Credit_History  \
0                0.0         NaN             360.0             1.0   
1             1508.0       128.0             360.0             1.0   
2                0.0        66.0             360.0             1.0   
3             2358.0       120.0             360.0             1.0   
4                0.0       141.0             360.0             1.0   

  Property_Area Loan_Status  
0         Urban           Y  
1         Rural           N  


# Target Distribution Plot
plt.figure(figsize=(6,4))
sns.countplot(x=target_col, data=df, palette='viridis')
plt.title("Loan Approval Distribution (Target)")
plt.show()

In [3]:
# 3. Preprocessing
# Fill missing values
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].fillna(df[col].mode()[0])
    else:
        df[col] = df[col].fillna(df[col].median())

# Encode Categorical features into numbers
le = LabelEncoder()
for col in df.select_dtypes(include=['object']).columns:
    df[col] = le.fit_transform(df[col])

X = df.drop(target_col, axis=1)
y = df[target_col]

# Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# PCA (Dimensionality Reduction)
pca = PCA(n_components=0.95)
X_pca = pca.fit_transform(X_scaled)

print(f"Original features: {X.shape[1]}")
print(f"PCA features: {X_pca.shape[1]}")

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X_pca, y, test_size=0.2, random_state=42, stratify=y
)

Original features: 11
PCA features: 10


In [4]:
# Classification models matching your previous flow
models = [
    ('Logistic', GridSearchCV(LogisticRegression(), {'C': [0.1, 1, 10]}, cv=5)),
    ('KNN', GridSearchCV(KNeighborsClassifier(), {'n_neighbors': [3, 5, 7]}, cv=5)),
    ('RandomForest', GridSearchCV(RandomForestClassifier(), {'n_estimators': [50, 100]}, cv=3))
]

results = []

for name, grid in models:
    grid.fit(X_train, y_train)
    best_model = grid.best_estimator_
    y_pred = best_model.predict(X_test)

    results.append({
        'Model': name,
        'Best Params': grid.best_params_,
        'CV Best Score': round(grid.best_score_, 4),
        'Test Accuracy': round(accuracy_score(y_test, y_pred), 4)
    })

# Final Results
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by='Test Accuracy', ascending=False).reset_index(drop=True)

print("\nModel Comparison")
print(results_df)

print("\nBest model on test data:", results_df.loc[0, 'Model'])


Model Comparison
          Model            Best Params  CV Best Score  Test Accuracy
0      Logistic             {'C': 0.1}         0.7963         0.8618
1           KNN     {'n_neighbors': 5}         0.7862         0.8618
2  RandomForest  {'n_estimators': 100}         0.7596         0.8130

Best model on test data: Logistic
